In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# Hyperparameters
batch_size = 8
num_class = 264
learning_rate = 0.001
num_epochs = 20

S = 13 # Grid size
B = 2 # Number of bounding boxes per grid
C = 264 # Number of food classes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#Create a CNN class. Extract visual features from the input image
class CNNBackbone(nn.Module):
    def __init__(self):
        super(CNNBackbone, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=1, padding=1), #3 input channels - RGB image
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2), # Halves spatial size

            nn.Conv2d(16, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  

            nn.Conv2d(128, 256, 3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2), # Final output: [B, 256, 13, 13]
        )

    def forward(self, x):
        return self.features(x)

# Detection Head. Predicts Bounding box coordinates, objectness score, class probabilities
class ModelHead(nn.Module):
    def __init__(self, S=13, B=2, C=264):
        super(ModelHead, self).__init__()
        self.S = S
        self.B = B
        self.C = C
        self.output_dim = B * (5 + C) # Each box: (x, y, w, h, obj)

        self.head = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(512, self.output_dim, 1), # Output: [B, 538, 13, 13]
        )

    def forward(self, X):
        x = self.head(X)
        x = x.permute(0, 2, 3, 1) # [B, 13, 13, 538]
        return x

# Combined Model
class Model(nn.Module):
    """
    This class combines the pieces:
        Calls the backbone    
        Feeds the output into the head        
        Returns final predictions
    """
    def __init__(self, S=13, B=2, C=264):
        super(Model, self).__init__()
        self.backbone = CNNBackbone()
        self.head = ModelHead(S, B, C)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

model = Model(S=S, B=B, C=C)
print("Model output shape:", model(torch.randn(2, 3, 416, 416)).shape)

Model output shape: torch.Size([2, 13, 13, 538])


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import CocoDetection
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from PIL import Image
import copy

# Custom CNN Backbone
class CustomCNNBackbone(nn.Module):
    def __init__(self):
        super(CustomCNNBackbone, self).__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.body(x)

class BackboneWithFPN(nn.Module):
    def __init__(self):
        super(BackboneWithFPN, self).__init__()
        self.backbone = CustomCNNBackbone()
        self.out_channels = 128

    def forward(self, x):
        return {"0": self.backbone(x)}

# Collate function for variable targets
def collate_fn(batch):
    return tuple(zip(*batch))

# Save checkpoint
def save_checkpoint(model, epoch, path="best_model_cnn.pth"):
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()}, path)

# Training function
def train_model_cnn(train_dir, val_dir, num_classes, num_epochs=20, patience=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Use precomputed mean and std to save time
    mean = [0.7274495959281921, 0.686897873878479, 0.6388996839523315]
    std = [0.26163002848625183, 0.286141037940979, 0.34688690304756165]
    print(f"Using precomputed Mean: {mean}, Std: {std}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    train_dataset = CocoDetection(
        root=os.path.join(train_dir, "images"),
        annFile=os.path.join(train_dir, "coco_train.json"),
        transform=transform
    )
    val_dataset = CocoDetection(
        root=os.path.join(val_dir, "images"),
        annFile=os.path.join(val_dir, "coco_val.json"),
        transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

    backbone = BackboneWithFPN()
    anchor_generator = AnchorGenerator(sizes=((32, 64, 128, 256),), aspect_ratios=((0.5, 1.0, 2.0),))
    roi_pooler = MultiScaleRoIAlign(featmap_names=['0'], output_size=7, sampling_ratio=2)

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler
    )
    model.to(device)

    optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

    best_loss = float("inf")
    best_model = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for images, targets in train_loader:
            processed_targets = []
            filtered_images = []

            for img, t in zip(images, targets):
                boxes = []
                labels = []
                for obj in t:
                    bbox = obj["bbox"]
                    x, y, w, h = bbox
                    boxes.append([x, y, x + w, y + h])
                    labels.append(obj["category_id"])
                if len(boxes) > 0:
                    filtered_images.append(img.to(device))
                    processed_targets.append({
                        "boxes": torch.tensor(boxes, dtype=torch.float32, device=device),
                        "labels": torch.tensor(labels, dtype=torch.int64, device=device)
                    })

            if len(processed_targets) == 0:
                continue

            loss_dict = model(filtered_images, processed_targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                processed_targets = []
                filtered_images = []
                for img, t in zip(images, targets):
                    boxes = []
                    labels = []
                    for obj in t:
                        bbox = obj["bbox"]
                        x, y, w, h = bbox
                        boxes.append([x, y, x + w, y + h])
                        labels.append(obj["category_id"])
                    if len(boxes) > 0:
                        filtered_images.append(img.to(device))
                        processed_targets.append({
                            "boxes": torch.tensor(boxes, dtype=torch.float32, device=device),
                            "labels": torch.tensor(labels, dtype=torch.int64, device=device)
                        })

                if len(processed_targets) == 0:
                    continue

                loss_dict = model(filtered_images, processed_targets)
                val_loss += sum(loss for loss in loss_dict.values()).item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_model = copy.deepcopy(model.state_dict())
            save_checkpoint(model, epoch + 1)
            print("Model improved. Checkpoint saved.")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s).")

        if epochs_no_improve >= patience:
            print("Early stopping.")
            break

    if best_model:
        model.load_state_dict(best_model)
        print("Loaded best model.")

    return model

# Entry point
if __name__ == "__main__":
    model = train_model_cnn(
        train_dir="/home/aojwang/data/yolo_dataset/train",
        val_dir="/home/aojwang/data/yolo_dataset/val",
        num_classes=265,
        num_epochs=5,
        patience=3
    )


Using precomputed Mean: [0.7274495959281921, 0.686897873878479, 0.6388996839523315], Std: [0.26163002848625183, 0.286141037940979, 0.34688690304756165]
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


In [13]:
!nvidia-smi

Wed Jun 25 17:00:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    Off |   00000000:00:1E.0 Off |                    0 |
|  0%   38C    P0             62W /  300W |    7785MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----